# Phase 2.2 — Data Exploration Notebook

All panels use `load_prices(final_holdout=False)`. Final holdout 2024-01-01 → 2025-06-30 is never read.

**Run order**: Kernel → Restart & Run All. Every cell must pass with zero errors before proceeding to Phase 3.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data_loader import FinalHoldoutDate, load_prices

sns.set_theme(style="whitegrid", rc={"figure.figsize": (14, 7)})

UNIVERSE_CSV = ROOT / "data" / "raw" / "universe_frozen.csv"
SECTOR_CSV = ROOT / "docs" / "nifty50_sector_map.csv"
STUDY_START = pd.Timestamp("2015-01-01")
DEV_END = pd.Timestamp("2023-12-31")
ILLIQUID_VOL_THRESH = 10_000

print("ROOT =", ROOT)
print("FinalHoldoutDate =", FinalHoldoutDate.date())
universe = pd.read_csv(UNIVERSE_CSV)
sector_map = pd.read_csv(SECTOR_CSV)
tickers = sorted(universe["ticker"].tolist())
print(f"N tickers = {len(tickers)}")

prices = load_prices(tickers, start_date=STUDY_START, end_date=DEV_END, final_holdout=False)
assert not prices.empty, "price frame empty"
assert prices.index.max() < FinalHoldoutDate, "holdout LEAK in notebook price load"
close = prices["Close"]
volume = prices["Volume"]
print(f"price shape = {prices.shape}, date range {prices.index.min().date()} -> {prices.index.max().date()}")
print("\nAll panels below are computed strictly inside the dev window (no holdout data).")

## Panel 6.1 — NaN structure heatmap (ticker × year) for Close

Any cell with >1% NaN in a single year should be flagged. F1 guarantees the 9-yr aggregate is <5%, but single-year spikes are worth a visual check.

In [ ]:
nan_by_ty = pd.DataFrame(index=close.columns)
for year, g in close.groupby(close.index.year):
    nan_by_ty[str(year)] = g.isna().mean() * 100

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(nan_by_ty, cmap="YlOrRd", annot=False, cbar_kws={"label": "% NaN rows / year"}, ax=ax)
ax.set_title(f"Panel 6.1: Close NaN fraction by ticker × year (N={len(close.columns)} tickers, {nan_by_ty.shape[1]} yrs)")
ax.set_xlabel("Year")
ax.set_ylabel("Ticker")
plt.tight_layout()
plt.show()

spike_mask = (nan_by_ty > 1.0).any(axis=1)
spike_tickers = nan_by_ty.loc[spike_mask]
if spike_tickers.empty:
    print("[ok  ] No ticker had >1% Close NaN in any single year across 2015-2023 dev window.")
else:
    print(f"[WARN] {len(spike_tickers)} ticker(s) had >1% Close NaN in at least 1 year:")
    print(spike_tickers.round(2).to_string())

## Panel 6.2 — Full-range CAGR bar chart grouped by sector

CAGR computed from log-return aggregation over the exact 2015-01-01 → 2023-12-29 dev window. Grouped by `sector_provisional` column.

In [ ]:
def annualized_cagr_log(series: pd.Series) -> float:
    s = series.dropna()
    if len(s) < 2:
        return np.nan
    days = (s.index[-1] - s.index[0]).days
    if days < 30:
        return np.nan
    total_log = np.log(s.iloc[-1] / s.iloc[0])
    return float(np.exp(total_log * (365.25 / days)) - 1.0)

cagr_s = close.apply(annualized_cagr_log).rename("cagr")
cagr_df = cagr_s.reset_index().rename(columns={"index": "ticker", "Close": "ticker"})
cagr_df = cagr_s.reset_index()
cagr_df.columns = ["ticker", "cagr"]
sector_slim = sector_map[["ticker", "sector_provisional"]].copy()
sector_slim.columns = ["ticker", "sector"]
cagr_df = cagr_df.merge(sector_slim, on="ticker", how="left")
cagr_df = cagr_df.sort_values(["sector", "cagr"], ascending=[True, False]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(16, 8))
sns.barplot(data=cagr_df, x="ticker", y="cagr", hue="sector", dodge=False, ax=ax)
ax.set_title("Panel 6.2: Annualized CAGR 2015-01 -> 2023-12 by ticker, grouped by sector (log-return method)")
ax.set_ylabel("Annualized CAGR (decimal)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v*100:.0f}%"))
plt.xticks(rotation=75, ha="right")
ax.legend(title="Sector", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

print(f"CAGR range: min = {cagr_df['cagr'].min()*100:+.2f}% ({cagr_df.loc[cagr_df['cagr'].idxmin(), 'ticker']}), "
      f"max = {cagr_df['cagr'].max()*100:+.2f}% ({cagr_df.loc[cagr_df['cagr'].idxmax(), 'ticker']}), "
      f"median = {cagr_df['cagr'].median()*100:+.2f}%")

## Panel 6.3 — Annualized daily vol violin plots by sector × year (2015 / 2020 / 2023)

In [ ]:
log_ret = np.log(close / close.shift(1))
rows = []
target_years = [2015, 2020, 2023]
for year in target_years:
    yr = log_ret.loc[log_ret.index.year == year]
    if yr.empty:
        continue
    vol_s = yr.std() * np.sqrt(252)
    for tkr, v in vol_s.items():
        sec = sector_slim.loc[sector_slim["ticker"] == tkr, "sector"].iloc[0] if tkr in sector_slim["ticker"].values else "Unknown"
        rows.append({"year": int(year), "ticker": tkr, "sector": sec, "ann_vol": float(v)})
vol_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(16, 8))
sns.violinplot(data=vol_df, x="sector", y="ann_vol", hue="year", split=False, inner="quartile", ax=ax)
ax.set_title("Panel 6.3: Annualized daily log-return volatility distribution by sector — 2015 vs 2020 vs 2023")
ax.set_ylabel("Annualized volatility (decimal)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v*100:.0f}%"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

for year in target_years:
    sub = vol_df[vol_df["year"] == year]
    if not sub.empty:
        print(f"{year}: median ann. vol = {sub['ann_vol'].median()*100:.1f}%, "
              f"p90 = {sub['ann_vol'].quantile(0.9)*100:.1f}%, p10 = {sub['ann_vol'].quantile(0.1)*100:.1f}%")

## Panel 6.4 — Cross-ticker daily return correlation matrix heatmap (ordered by sector for clustering)

In [ ]:
sector_order = sector_slim.set_index("ticker").loc[close.columns, "sector"]
corr_mat = log_ret.corr()
tickers_ordered = sector_order.sort_values().index.tolist()
corr_sorted = corr_mat.loc[tickers_ordered, tickers_ordered]

sectors_sorted = sector_order.loc[tickers_ordered]
uniq_sec = sectors_sorted.unique().tolist()
pal = sns.husl_palette(len(uniq_sec), s=0.5)
sec_color = dict(zip(uniq_sec, pal))
col_colors = sectors_sorted.map(sec_color)

g = sns.clustermap(
    corr_sorted,
    method="average",
    row_cluster=False,
    col_cluster=False,
    col_colors=col_colors,
    row_colors=col_colors,
    cmap="RdBu_r",
    vmin=-0.2,
    vmax=1.0,
    figsize=(18, 16),
    cbar_kws={"label": "Pearson corr of daily log return"},
)
g.fig.suptitle(f"Panel 6.4: {len(tickers_ordered)} × {len(tickers_ordered)} return correlation — tickers sorted by sector (color bands)", y=1.02)
g.ax_heatmap.set_xticks([])
g.ax_heatmap.set_yticks([])
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=sec_color[s], label=s) for s in uniq_sec]
g.ax_col_dendrogram.legend(handles=legend_handles, title="Sector", loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.show()

corr_vals = corr_sorted.values[np.triu_indices_from(corr_sorted.values, k=1)]
print(f"Pairwise correlations: mean = {np.nanmean(corr_vals):.3f}, median = {np.nanmedian(corr_vals):.3f}, "
      f"min = {np.nanmin(corr_vals):.3f}, max = {np.nanmax(corr_vals):.3f}")

## Panel 6.5 — Liquidity-by-year: count days per ticker per year with volume < threshold

In [ ]:
illiquid_flag = (volume == 0) | (volume < ILLIQUID_VOL_THRESH)
liq_rows = []
for year, g in illiquid_flag.groupby(illiquid_flag.index.year):
    total_days = len(g)
    if total_days == 0:
        continue
    frac_ill = g.mean()
    for tkr, f in frac_ill.items():
        liq_rows.append({"year": int(year), "ticker": tkr, "illiquid_frac": float(f)})
liq_df = pd.DataFrame(liq_rows)

fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(data=liq_df, x="year", y="illiquid_frac", ax=ax, color="#4C72B0")
sns.stripplot(data=liq_df, x="year", y="illiquid_frac", ax=ax, size=3, color="#2b2b2b", alpha=0.4)
ax.axhline(0.02, color="#c0392b", ls="--", lw=1.5, label="F2 pass threshold = 2% (9-yr avg)")
ax.set_title(f"Panel 6.5: Illiquid-day fraction per ticker per year (vol < {ILLIQUID_VOL_THRESH:,} shares). Red line = F2 2% aggregate threshold.")
ax.set_ylabel("Fraction of trading days illiquid")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v*100:.1f}%"))
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

worst = liq_df.loc[liq_df["illiquid_frac"].idxmax()]
print(f"Worst single-year illiquid spike: {worst['ticker']} @ {int(worst['year'])} = {worst['illiquid_frac']*100:.2f}% trading days illiquid")
print(f"Ticker-year combos >5% illiquid: {int((liq_df['illiquid_frac'] > 0.05).sum())} / {len(liq_df)}")
print(f"Ticker-year combos >10% illiquid: {int((liq_df['illiquid_frac'] > 0.10).sum())} / {len(liq_df)}")

## Panel 6.6 — Sector summary table + export to `data/processed/sector_summary_dev_window.csv`

In [ ]:
sec_stats_rows = []
for sec, grp in cagr_df.groupby("sector"):
    sec_tickers = grp["ticker"].tolist()
    n_sec = len(sec_tickers)
    close_sec = close[sec_tickers]
    vol_sec = volume[sec_tickers]
    cagr_mean = grp["cagr"].mean()
    lr_sec = np.log(close_sec / close_sec.shift(1))
    vol_ann_mean = (lr_sec.std() * np.sqrt(252)).mean()
    illiquid_flag_sec = (vol_sec == 0) | (vol_sec < ILLIQUID_VOL_THRESH)
    median_illiquid_by_ticker = illiquid_flag_sec.mean(axis=0).median()
    ffr = universe.set_index("ticker").loc[sec_tickers, "free_float_rank"] if "free_float_rank" in universe.columns else pd.Series(dtype=float)
    mean_ffr = float(ffr.mean()) if not ffr.empty else np.nan
    sec_stats_rows.append({
        "sector": sec,
        "n_tickers": int(n_sec),
        "cagr_mean_ann": float(cagr_mean),
        "ann_vol_mean": float(vol_ann_mean),
        "free_float_rank_mean": mean_ffr,
        "median_illiquid_frac_9yr": float(median_illiquid_by_ticker),
    })
sector_summary = pd.DataFrame(sec_stats_rows).sort_values("n_tickers", ascending=False).reset_index(drop=True)

out_dir = ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "sector_summary_dev_window.csv"
sector_summary.to_csv(out_path, index=False)

display_cols = {
    "cagr_mean_ann": "{:.2%}".format,
    "ann_vol_mean": "{:.2%}".format,
    "median_illiquid_frac_9yr": "{:.3%}".format,
    "free_float_rank_mean": "{:.1f}".format,
}
styled = sector_summary.style.format(display_cols)
print(f"Sector summary shape = {sector_summary.shape}; exported to: {out_path}")
from IPython.display import display
display(styled)

## Notebook Completion Signature

Run this last cell to log final diagnostics used as Phase 2 gate evidence.

In [ ]:
print("=== Notebook 01_data_exploration.ipynb RUN COMPLETE ===")
print(f"Ticker N:                 {len(tickers)}")
print(f"Dev window:               {STUDY_START.date()} -> {DEV_END.date()}")
print(f"Final holdout embargo:    {FinalHoldoutDate.date()} -> 2025-06-30 (NOT LOADED)")
print(f"Close NaN any >1%/yr:     {bool(spike_mask.any()) if 'spike_mask' in dir() else 'N/A'}")
print(f"Sectors in summary:       {len(sector_summary)}")
print(f"Sector summary exported:  {out_path.exists()}")
print("Restart & Run All status: OK (this cell would error if any prior cell failed)")